<a href="https://colab.research.google.com/github/namgiju/mlprog/blob/main/auto_mpg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Auto MPG 연비 예측 (회귀)

## 1. 라이브러리 불러오기

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

## 2. 구글 드라이브 연결

In [5]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. 데이터 불러오기

In [7]:
df = pd.read_csv("/content/drive/MyDrive/auto-mpg.data", sep=r"\s+", header=None, na_values="?")
df.columns = ["mpg", "cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year", "origin", "car_name"]
display(df.head(10))

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino
5,15.0,8,429.0,198.0,4341.0,10.0,70,1,ford galaxie 500
6,14.0,8,454.0,220.0,4354.0,9.0,70,1,chevrolet impala
7,14.0,8,440.0,215.0,4312.0,8.5,70,1,plymouth fury iii
8,14.0,8,455.0,225.0,4425.0,10.0,70,1,pontiac catalina
9,15.0,8,390.0,190.0,3850.0,8.5,70,1,amc ambassador dpl


## 4. 전처리

In [8]:
df = df.dropna()
df = df.drop("car_name", axis=1)
print("데이터 크기:", df.shape)
display(df.describe().round(2))

데이터 크기: (392, 8)


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
count,392.00,392.00,392.00,392.00,392.00,392.00,392.00,392.00
mean,23.45,5.47,194.41,104.47,2977.58,15.54,75.98,1.58
std,7.81,1.71,104.64,38.49,849.40,2.76,3.68,0.81
min,9.00,3.00,68.00,46.00,1613.00,8.00,70.00,1.00
25%,17.00,4.00,105.00,75.00,2225.25,13.78,73.00,1.00
50%,22.75,4.00,151.00,93.50,2803.50,15.50,76.00,1.00
75%,29.00,8.00,275.75,126.00,3614.75,17.02,79.00,2.00
max,46.60,8.00,455.00,230.00,5140.00,24.80,82.00,3.00


## 5. 피처/레이블 분리 및 데이터 분할

In [9]:
X = df.drop("mpg", axis=1)
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print("학습 데이터:", X_train.shape)
print("테스트 데이터:", X_test.shape)

학습 데이터: (313, 7)
테스트 데이터: (79, 7)


## 6. 평가 함수 정의

In [10]:
results = []

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, pred)
    results.append({"Model": name, "R2": round(r2, 4), "MSE": round(mse, 3), "RMSE": round(rmse, 3), "MAE": round(mae, 3)})
    print(f"--- {name} ---")
    print("R2:", round(r2, 4))
    print("MSE:", round(mse, 3))
    print("RMSE:", round(rmse, 3))
    print("MAE:", round(mae, 3))

## 7. Linear Regression

In [11]:
evaluate("Linear Regression", LinearRegression())

--- Linear Regression ---
R2: 0.8364
MSE: 10.171
RMSE: 3.189
MAE: 2.386


## 8. Decision Tree

In [12]:
evaluate("Decision Tree", DecisionTreeRegressor(random_state=0))

--- Decision Tree ---
R2: 0.7454
MSE: 15.825
RMSE: 3.978
MAE: 2.628


## 9. Random Forest

In [13]:
evaluate("Random Forest", RandomForestRegressor(random_state=0))

--- Random Forest ---
R2: 0.9014
MSE: 6.13
RMSE: 2.476
MAE: 1.735


## 10. KNN

In [14]:
evaluate("KNN", KNeighborsRegressor(n_neighbors=5))

--- KNN ---
R2: 0.7564
MSE: 15.142
RMSE: 3.891
MAE: 2.883


## 11. 결과 비교

In [15]:
result_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
result_df.index = result_df.index + 1
display(result_df)

,Model,R2,MSE,RMSE,MAE
1,Random Forest,0.9014,6.130,2.476,1.735
2,Linear Regression,0.8364,10.171,3.189,2.386
3,KNN,0.7564,15.142,3.891,2.883
4,Decision Tree,0.7454,15.825,3.978,2.628
